# PySpark reference 11: Spark SQL and the catalog

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-11-spark-sql-and-catalog.ipynb)

Running SQL with parameters and DataFrame arguments, temporary and global views, the catalog API, managed and external tables, DDL and DML, CTEs and subqueries, generators and lambdas in SQL, PIVOT, dynamic SQL and SQL functions. Practise it in [PySpark practice 11: Spark SQL and the catalog](pyspark-practice-11-spark-sql-and-catalog.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose. The shared tables are also registered as temporary views with the same names.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:110])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

import os

orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")
order_items.createOrReplaceTempView("order_items")
products.createOrReplaceTempView("products")

## Contents

| section | entries |
| --- | --- |
| [Running SQL](#running-sql) | [`spark.sql(sqlQuery, args=None, **kwargs)`](#sparksqlsqlquery-argsnone-kwargs), [Temporary and global views](#temporary-and-global-views), [Moving between SQL and DataFrames](#moving-between-sql-and-dataframes) |
| [The catalog](#the-catalog) | [`spark.catalog`](#sparkcatalog) |
| [Tables and DDL](#tables-and-ddl) | [Managed and external tables](#managed-and-external-tables), [Common DDL and DML](#common-ddl-and-dml) |
| [SQL features worth knowing](#sql-features-worth-knowing) | [CTEs, subqueries and lateral column aliases](#ctes-subqueries-and-lateral-column-aliases), [Generators, higher-order functions and PIVOT in SQL](#generators-higher-order-functions-and-pivot-in-sql), [Dynamic SQL: `IDENTIFIER()`, variables and `EXECUTE IMMEDIATE`](#dynamic-sql-identifier-variables-and-execute-immediate), [SQL functions: `CREATE FUNCTION`](#sql-functions-create-function), [Session settings with `SET`](#session-settings-with-set) |

## Running SQL

### `spark.sql(sqlQuery, args=None, **kwargs)`

Runs one SQL statement and returns its result as a DataFrame (queries) or an empty DataFrame (DDL and commands). The
result is lazy like any DataFrame, and SQL and DataFrame code compile to the same plans, so performance is identical.

| parameter | type | description |
| --- | --- | --- |
| `sqlQuery` | str | one SQL statement (no trailing statements separated by `;`) |
| `args` | dict or list, optional | **parameter values**: a dict for named markers `:name`, a list for positional markers `?`. Values are bound safely, never pasted into the text |
| `**kwargs` | DataFrame or values | **substitution**: `{name}` in the text is replaced by the DataFrame (as a temporary relation) or the value's string form |

**Returns:** a DataFrame.

**Notes:** use `args` for user-supplied values; `{name}` formatting with strings is plain text substitution and open
to SQL injection. Passing DataFrames through `{name}` avoids registering temporary views.

In [ ]:
spark.sql("SELECT status, count(*) AS orders FROM orders GROUP BY status ORDER BY status").show()
spark.sql("SELECT order_id, amount FROM orders WHERE amount > :min AND status = :status ORDER BY order_id",
          args={"min": 100, "status": "paid"}).show()
spark.sql("SELECT order_id FROM orders WHERE customer_id = ? ORDER BY order_id", args=[1]).show()
big = orders.where("amount > 200")
spark.sql("SELECT b.order_id, c.name FROM {big} b JOIN {c} c USING (customer_id) ORDER BY 1", big=big, c=customers).show()

### Temporary and global views

| method | visible to | lives until |
| --- | --- | --- |
| `df.createOrReplaceTempView(name)` | this `SparkSession` | the session ends or `spark.catalog.dropTempView(name)` |
| `df.createOrReplaceGlobalTempView(name)` | all sessions of the application, as `global_temp.name` | the application ends |
| `CREATE VIEW name AS ...` (SQL) | everyone using the catalog | dropped; stored in the metastore |

A view stores the **plan**, not the data: every query recomputes it unless the underlying DataFrame is cached.

**Notes:** `createTempView` (without `OrReplace`) fails if the name exists. Temporary views shadow tables with the same
name.

In [ ]:
orders.where("status = 'paid'").createOrReplaceTempView("paid_orders")
print("temp view rows:", spark.sql("SELECT count(*) FROM paid_orders").first()[0])
customers.createOrReplaceGlobalTempView("customers_g")
other = spark.newSession()
print("global view from another session:", other.sql("SELECT count(*) FROM global_temp.customers_g").first()[0])
print("temp view from another session exists:", other.catalog.tableExists("paid_orders"))

### Moving between SQL and DataFrames

| from | to | how |
| --- | --- | --- |
| DataFrame | SQL | `df.createOrReplaceTempView(name)` or `spark.sql("... {df} ...", df=df)` |
| SQL | DataFrame | `spark.sql(...)` returns one; `spark.table(name)` reads a table or view |
| SQL expression | Column | `F.expr("...")`, `df.selectExpr(...)`, `df.where("...")` |

**Notes:** mix freely: build complex logic in SQL and continue in Python, or the reverse.

In [ ]:
top = spark.sql("SELECT customer_id, sum(amount) AS revenue FROM orders GROUP BY customer_id")
top.where(F.col("revenue") > 300).join(spark.table("customers"), "customer_id").select("name", "revenue").orderBy(F.desc("revenue")).show()

## The catalog

### `spark.catalog`

The catalog API lists and manages databases, tables, views, functions and the cache.

| method | returns / does |
| --- | --- |
| `currentDatabase()`, `setCurrentDatabase(name)` | the default database for unqualified names |
| `listDatabases()`, `listTables(dbName=None)`, `listColumns(tableName)`, `listFunctions()` | metadata as lists of named tuples |
| `tableExists(name)`, `databaseExists(name)`, `functionExists(name)` | booleans |
| `getTable(name)` | name, database, `tableType` (`MANAGED`, `EXTERNAL`, `VIEW`, `TEMPORARY`), `isTemporary` |
| `dropTempView(name)`, `dropGlobalTempView(name)` | remove views |
| `cacheTable(name)`, `uncacheTable(name)`, `isCached(name)`, `clearCache()` | cache by name |
| `refreshTable(name)`, `recoverPartitions(name)` | reload file listings / discover partition folders |

**Notes:** `listTables(db)` also returns the session's temporary views (`isTemporary=True`); filter them out when
inventorying a database. Spark 3.4+ supports multiple catalogs (`catalog.database.table`); `currentCatalog()` and `setCurrentCatalog`
switch between them (Unity Catalog, Iceberg and Hive catalogs plug in here).

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS shop")
spark.catalog.setCurrentDatabase("shop")
orders.write.mode("overwrite").saveAsTable("shop.orders_tbl")
print("current:", spark.catalog.currentCatalog(), spark.catalog.currentDatabase())
print([(t.name, t.tableType, t.isTemporary) for t in spark.catalog.listTables("shop")])
print([c.name + ":" + c.dataType for c in spark.catalog.listColumns("shop.orders_tbl")])
print("exists:", spark.catalog.tableExists("shop.orders_tbl"), spark.catalog.tableExists("shop.nope"))
spark.catalog.setCurrentDatabase("default")

## Tables and DDL

### Managed and external tables

| | managed | external |
| --- | --- | --- |
| created by | `CREATE TABLE t (...) USING parquet` / `saveAsTable` | `CREATE TABLE t (...) USING parquet LOCATION '/path'` / `.option("path", p).saveAsTable` |
| data stored | under `spark.sql.warehouse.dir` | at the given location |
| `DROP TABLE` | deletes metadata **and data** | deletes metadata only; files stay |

**Use it for:** external tables when data is shared with other engines or must survive the table definition;
managed tables when the catalog owns the data's lifecycle.

In [ ]:
ext_path = f"{WORK_DIR}/ext_orders"
orders.write.mode("overwrite").parquet(ext_path)
spark.sql("DROP TABLE IF EXISTS ext_orders")
spark.sql(f"CREATE TABLE ext_orders USING parquet LOCATION '{ext_path}'")
spark.sql("DROP TABLE IF EXISTS managed_orders")
spark.sql("CREATE TABLE managed_orders USING parquet AS SELECT * FROM orders")
managed_path = spark.sql("DESCRIBE TABLE EXTENDED managed_orders").where("col_name = 'Location'").first().data_type.replace("file:", "")
print(spark.catalog.getTable("ext_orders").tableType, spark.catalog.getTable("managed_orders").tableType)
spark.sql("DROP TABLE ext_orders"); spark.sql("DROP TABLE managed_orders")
print("external files remain:", os.path.exists(ext_path), "| managed files remain:", os.path.exists(managed_path))

### Common DDL and DML

| statement | purpose |
| --- | --- |
| `CREATE TABLE t (id INT, day DATE) USING parquet PARTITIONED BY (day)` | define a table |
| `CREATE TABLE t AS SELECT ...` (CTAS) | create from a query |
| `INSERT INTO t SELECT ...` / `INSERT OVERWRITE t [PARTITION (day='...')] SELECT ...` | append / replace |
| `ALTER TABLE t ADD COLUMNS (note STRING)` | evolve the schema |
| `ALTER TABLE t RENAME TO t2`, `ALTER TABLE t SET TBLPROPERTIES (...)` | rename, properties |
| `DESCRIBE [EXTENDED] t`, `SHOW TABLES`, `SHOW PARTITIONS t`, `SHOW CREATE TABLE t` | inspect |
| `TRUNCATE TABLE t`, `DROP TABLE [IF EXISTS] t` | empty / remove |
| `MSCK REPAIR TABLE t` | register partition folders added outside Spark |

**Notes:** `INSERT OVERWRITE` with a static partition spec replaces only that partition; with dynamic partitions it
follows `spark.sql.sources.partitionOverwriteMode`.

In [ ]:
spark.sql("DROP TABLE IF EXISTS daily_orders")
spark.sql("CREATE TABLE daily_orders (order_id INT, amount DECIMAL(12,2), day DATE) USING parquet PARTITIONED BY (day)")
spark.sql("INSERT INTO daily_orders SELECT order_id, amount, to_date(order_ts) FROM orders")
spark.sql("INSERT OVERWRITE daily_orders PARTITION (day = '2026-01-05') SELECT order_id, amount * 2 FROM orders WHERE to_date(order_ts) = '2026-01-05'")
spark.sql("SHOW PARTITIONS daily_orders").show(3)
spark.sql("ALTER TABLE daily_orders ADD COLUMNS (note STRING)")
spark.sql("SELECT day, count(*) AS n, sum(amount) AS revenue FROM daily_orders GROUP BY day ORDER BY day").show(3)
print(spark.table("daily_orders").columns)

## SQL features worth knowing

### CTEs, subqueries and lateral column aliases

- **CTEs** (`WITH a AS (...), b AS (...) SELECT ...`) name intermediate results; Spark inlines them, so a CTE used twice
  is computed twice unless the plan reuses the exchange.
- **Subqueries:** scalar (`(SELECT max(...) FROM ...)`), `IN`, `EXISTS` / `NOT EXISTS`, and correlated subqueries that
  refer to the outer query; Spark rewrites them into joins.
- **Lateral column aliases** (Spark 3.4+): a later expression in the same `SELECT` can use an earlier alias.

In [ ]:
spark.sql("""
    WITH revenue AS (SELECT customer_id, sum(amount) AS total FROM orders GROUP BY customer_id),
         ranked AS (SELECT *, total * 100 / (SELECT sum(total) FROM revenue) AS pct FROM revenue)
    SELECT c.name, round(pct, 1) AS pct, pct > 20 AS major
    FROM ranked r JOIN customers c USING (customer_id)
    WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = r.customer_id AND o.status = 'returned')
    ORDER BY pct DESC
""").show()

### Generators, higher-order functions and PIVOT in SQL

| feature | syntax |
| --- | --- |
| explode in `FROM` | `FROM t LATERAL VIEW explode(arr) e AS x` or `SELECT explode(arr) AS x FROM t` |
| lambda functions | `transform(arr, x -> x * 2)`, `filter(arr, x -> x > 0)`, `aggregate(arr, 0, (acc, x) -> acc + x)` |
| pivot | `SELECT * FROM t PIVOT (sum(v) FOR k IN ('a', 'b'))` |
| unpivot | `SELECT * FROM t UNPIVOT (val FOR col IN (a, b))` |
| `GROUP BY ALL`, `FILTER (WHERE ...)` | see [aggregations](pyspark-reference-07-aggregations.ipynb#group-by-group-by-all-having-and-filter) |

In [ ]:
spark.sql("""
    SELECT customer_id, paid, shipped, returned
    FROM (SELECT customer_id, status, amount FROM orders)
    PIVOT (sum(amount) FOR status IN ('paid' AS paid, 'shipped' AS shipped, 'returned' AS returned))
    ORDER BY customer_id
""").show(3)
spark.sql("SELECT id, x, transform(array(1, 2, 3), v -> v * id) AS scaled FROM (SELECT 2 AS id) LATERAL VIEW explode(array('a', 'b')) e AS x").show()

### Dynamic SQL: `IDENTIFIER()`, variables and `EXECUTE IMMEDIATE`

| feature | example |
| --- | --- |
| `IDENTIFIER(:name)` | a table or column name passed as a parameter: `SELECT * FROM IDENTIFIER(:tbl)` |
| session variables (Spark 4) | `DECLARE VARIABLE cutoff DATE DEFAULT DATE'2026-01-10'`; `SET VAR cutoff = ...`; use `cutoff` in queries |
| `EXECUTE IMMEDIATE` (Spark 4) | run a SQL string built at run time, with `USING` for parameters |

**Use it for:** parameterized SQL jobs without string formatting, and SQL-only pipelines (dbt, notebooks) that need
dynamic table names.

In [ ]:
print(spark.sql("SELECT count(*) FROM IDENTIFIER(:tbl)", args={"tbl": "orders"}).first()[0])
spark.sql("DECLARE OR REPLACE VARIABLE cutoff DATE DEFAULT DATE'2026-01-10'")
print(spark.sql("SELECT count(*) FROM orders WHERE to_date(order_ts) >= cutoff").first()[0])
print(spark.sql("EXECUTE IMMEDIATE 'SELECT count(*) FROM orders WHERE status = ?' USING 'paid'").first()[0])

### SQL functions: `CREATE FUNCTION`

Spark 4 supports **SQL UDFs**: functions written in SQL (`RETURN expression` or `RETURN SELECT ...`) that the optimizer
can inline, unlike Python UDFs. `CREATE TEMPORARY FUNCTION` lasts for the session; without `TEMPORARY` the function is
stored in the catalog.

In [ ]:
spark.sql("CREATE OR REPLACE TEMPORARY FUNCTION gross(amount DECIMAL(12,2), rate DOUBLE) RETURNS DECIMAL(14,2) RETURN round(amount * (1 + rate), 2)")
spark.sql("SELECT order_id, amount, gross(amount, 0.2) AS gross FROM orders ORDER BY order_id LIMIT 3").show()

### Session settings with `SET`

`SET key=value` changes a runtime setting for the session, like `spark.conf.set`; `SET key` shows it and `RESET`
restores defaults.

In [ ]:
spark.sql("SET spark.sql.shuffle.partitions=8")
print(spark.conf.get("spark.sql.shuffle.partitions"))
spark.sql("RESET spark.sql.shuffle.partitions")
spark.conf.set("spark.sql.shuffle.partitions", "4")

**Next:** practise these entries in [PySpark practice 11: Spark SQL and the catalog](pyspark-practice-11-spark-sql-and-catalog.ipynb).